<a href="https://colab.research.google.com/github/Ayazmj/northstar-database-analytics-coursework/blob/main/notebooks/03_python_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 Python Data Processing and Feature Engineering

## Project Title
Integrated Database and Analytics Solution for NorthStar Urban Mobility and Logistics

## Purpose
This notebook performs Python-based data processing for the NorthStar dataset. It uses Pandas and NumPy to clean operational datasets, handle missing values, standardise column names, convert date/time fields, create analytical features, and export cleaned datasets for SQL in R, R analytics, and MongoDB development.

## Coursework Relevance
This notebook supports the Python data processing section of the coursework by demonstrating:
- Pandas-based data loading and inspection
- Missing value handling
- Duplicate checking
- Column standardisation
- Date/time conversion
- NumPy-based feature engineering
- Exporting cleaned datasets for later notebooks

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os

data_path = "/content/drive/MyDrive/NorthStar_Database_Analytics/data "
output_path = "/content/drive/MyDrive/NorthStar_Database_Analytics/outputs/"
cleaned_output_path = output_path + "cleaned_data/"

os.makedirs(output_path, exist_ok=True)
os.makedirs(cleaned_output_path, exist_ok=True)

print("Data path:", data_path)
print("Output path:", output_path)
print("Cleaned output path:", cleaned_output_path)

Data path: /content/drive/MyDrive/NorthStar_Database_Analytics/data 
Output path: /content/drive/MyDrive/NorthStar_Database_Analytics/outputs/
Cleaned output path: /content/drive/MyDrive/NorthStar_Database_Analytics/outputs/cleaned_data/


In [3]:
files = {
    "customers": "customers.csv",
    "orders": "orders.csv",
    "deliveries": "deliveries.csv",
    "drivers": "drivers.csv",
    "vehicles": "vehicles.csv",
    "hubs": "hubs.csv",
    "complaints": "complaints.csv",
    "incidents": "incidents.csv",
    "app_events": "app_events.csv",
    "data_dictionary": "data_dictionary.csv"
}

datasets = {}

for name, filename in files.items():
    file_path = os.path.join(data_path, filename)

    if os.path.exists(file_path):
        datasets[name] = pd.read_csv(file_path)
        print(f"{name} loaded successfully: {datasets[name].shape}")
    else:
        print(f"{name} NOT FOUND: {file_path}")

customers loaded successfully: (650, 9)
orders loaded successfully: (1250, 11)
deliveries loaded successfully: (950, 13)
drivers loaded successfully: (170, 8)
vehicles loaded successfully: (120, 8)
hubs loaded successfully: (8, 5)
complaints loaded successfully: (320, 10)
incidents loaded successfully: (280, 7)
app_events loaded successfully: (640, 10)
data_dictionary loaded successfully: (9, 3)


In [4]:
customers_clean = datasets["customers"].copy()
orders_clean = datasets["orders"].copy()
deliveries_clean = datasets["deliveries"].copy()
drivers_clean = datasets["drivers"].copy()
vehicles_clean = datasets["vehicles"].copy()
hubs_clean = datasets["hubs"].copy()
complaints_clean = datasets["complaints"].copy()
incidents_clean = datasets["incidents"].copy()
app_events_clean = datasets["app_events"].copy()

cleaned_datasets = {
    "customers": customers_clean,
    "orders": orders_clean,
    "deliveries": deliveries_clean,
    "drivers": drivers_clean,
    "vehicles": vehicles_clean,
    "hubs": hubs_clean,
    "complaints": complaints_clean,
    "incidents": incidents_clean,
    "app_events": app_events_clean
}

print("Cleaning copies created.")

Cleaning copies created.


In [5]:
for name, df in cleaned_datasets.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

print("Column names standardised.")

Column names standardised.


In [6]:
before_cleaning_summary = []

for name, df in cleaned_datasets.items():
    before_cleaning_summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values_before": df.isnull().sum().sum(),
        "duplicate_rows_before": df.duplicated().sum()
    })

before_cleaning_df = pd.DataFrame(before_cleaning_summary)
before_cleaning_df

,dataset,rows,columns,missing_values_before,duplicate_rows_before
0,customers,650,9,33,0
1,orders,1250,11,25,0
2,deliveries,950,13,33,0
3,drivers,170,8,7,0
4,vehicles,120,8,4,0
5,hubs,8,5,0,0
6,complaints,320,10,16,0
7,incidents,280,7,17,0
8,app_events,640,10,144,0


In [7]:
missing_details = []

for name, df in cleaned_datasets.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()
        if missing_count > 0:
            missing_details.append({
                "dataset": name,
                "column": column,
                "missing_count": missing_count,
                "missing_percentage": round((missing_count / len(df)) * 100, 2)
            })

missing_details_df = pd.DataFrame(missing_details)
missing_details_df.sort_values(by="missing_count", ascending=False)

,dataset,column,missing_count,missing_percentage
9,app_events,order_id,144,22.50
2,orders,booking_channel,25,2.00
0,customers,loyalty_score,20,3.08
3,deliveries,delivery_completed_at,19,2.00
8,incidents,resolved_hours,17,6.07
7,complaints,compensation_amount,16,5.00
4,deliveries,customer_rating_post_delivery,14,1.47
1,customers,preferred_channel,13,2.00
5,drivers,training_score,7,4.12
6,vehicles,battery_health_pct,4,3.33


## Data Quality Issues Identified

The initial Python processing stage identified missing values across several operational datasets. These missing values need to be handled before performing SQL queries, R analytics, and MongoDB document modelling. Missing categorical values are handled using the label `Unknown`, while missing numerical values are handled using the median to reduce the effect of outliers.

In [8]:
for name, df in cleaned_datasets.items():
    for column in df.columns:
        if df[column].dtype == "object":
            df[column] = df[column].fillna("Unknown")
        else:
            df[column] = df[column].fillna(df[column].median())

print("Missing values handled.")

Missing values handled.


In [9]:
after_missing_summary = []

for name, df in cleaned_datasets.items():
    after_missing_summary.append({
        "dataset": name,
        "missing_values_after": df.isnull().sum().sum()
    })

after_missing_df = pd.DataFrame(after_missing_summary)
after_missing_df

,dataset,missing_values_after
0,customers,0
1,orders,0
2,deliveries,0
3,drivers,0
4,vehicles,0
5,hubs,0
6,complaints,0
7,incidents,0
8,app_events,0


In [10]:
for name, df in cleaned_datasets.items():
    for column in df.columns:
        if "date" in column or "time" in column or "timestamp" in column:
            df[column] = pd.to_datetime(df[column], errors="coerce")

print("Date/time columns converted where possible.")

Date/time columns converted where possible.


In [11]:
for name, df in cleaned_datasets.items():
    date_cols = [col for col in df.columns if "date" in col or "time" in col or "timestamp" in col]

    if date_cols:
        print("\n" + "="*60)
        print(name.upper())
        print("="*60)
        print(df[date_cols].dtypes)


CUSTOMERS
signup_date    datetime64[ns]
dtype: object

DELIVERIES
dispatch_time    datetime64[ns]
dtype: object

VEHICLES
commission_date    datetime64[ns]
dtype: object

APP_EVENTS
event_timestamp    datetime64[ns]
dtype: object


In [12]:
for name, df in cleaned_datasets.items():
    delay_cols = [col for col in df.columns if "delay" in col]

    for col in delay_cols:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col + "_flag"] = np.where(df[col] > 0, 1, 0)

print("Delay flags created where delay columns exist.")

Delay flags created where delay columns exist.


In [13]:
failure_keywords = ["failed", "failure", "cancelled", "canceled", "missed", "late", "exception"]

for name, df in cleaned_datasets.items():
    status_cols = [col for col in df.columns if "status" in col]

    for col in status_cols:
        df[col + "_failure_flag"] = df[col].astype(str).str.lower().apply(
            lambda x: 1 if any(keyword in x for keyword in failure_keywords) else 0
        )

print("Failure flags created from status columns.")

Failure flags created from status columns.


In [14]:
severity_cols = [col for col in complaints_clean.columns if "severity" in col]

for col in severity_cols:
    complaints_clean[col + "_high_flag"] = complaints_clean[col].astype(str).str.lower().apply(
        lambda x: 1 if x in ["high", "critical", "severe"] else 0
    )

print("Complaint severity flags created.")

Complaint severity flags created.


In [15]:
event_cols = [col for col in app_events_clean.columns if "event" in col or "type" in col or "status" in col]

for col in event_cols:
    app_events_clean[col + "_issue_flag"] = app_events_clean[col].astype(str).str.lower().apply(
        lambda x: 1 if any(keyword in x for keyword in failure_keywords) else 0
    )

print("App event issue flags created where possible.")

App event issue flags created where possible.


In [16]:
processing_summary = []

for name, df in cleaned_datasets.items():
    processing_summary.append({
        "dataset": name,
        "rows_after_cleaning": df.shape[0],
        "columns_after_cleaning": df.shape[1],
        "missing_values_after_cleaning": df.isnull().sum().sum(),
        "duplicate_rows_after_cleaning": df.duplicated().sum()
    })

processing_summary_df = pd.DataFrame(processing_summary)
processing_summary_df

,dataset,rows_after_cleaning,columns_after_cleaning,missing_values_after_cleaning,duplicate_rows_after_cleaning
0,customers,650,10,0,0
1,orders,1250,11,0,0
2,deliveries,950,14,0,0
3,drivers,170,8,0,0
4,vehicles,120,9,0,0
5,hubs,8,5,0,0
6,complaints,320,12,0,0
7,incidents,280,8,0,0
8,app_events,640,14,0,0


In [17]:
for name, df in cleaned_datasets.items():
    df.to_csv(cleaned_output_path + f"{name}_cleaned.csv", index=False)

before_cleaning_df.to_csv(output_path + "before_cleaning_summary.csv", index=False)
missing_details_df.to_csv(output_path + "missing_values_by_column.csv", index=False)
processing_summary_df.to_csv(output_path + "python_processing_summary.csv", index=False)

print("Cleaned datasets and processing summaries exported successfully.")

Cleaned datasets and processing summaries exported successfully.


## Python Processing Summary

The Python processing notebook prepared the NorthStar datasets for later SQL, R analytics, and MongoDB development. The processing included standardising column names, handling missing values, converting date/time fields, creating delay and failure indicators, and exporting cleaned datasets.

The cleaned outputs will be reused in the SQL in R, R analytics, MongoDB development, and query optimisation notebooks. This supports reproducibility because each later notebook can load consistent cleaned datasets from the outputs folder.

In [18]:
missing_details_df.sort_values(by="missing_count", ascending=False)

,dataset,column,missing_count,missing_percentage
9,app_events,order_id,144,22.50
2,orders,booking_channel,25,2.00
0,customers,loyalty_score,20,3.08
3,deliveries,delivery_completed_at,19,2.00
8,incidents,resolved_hours,17,6.07
7,complaints,compensation_amount,16,5.00
4,deliveries,customer_rating_post_delivery,14,1.47
1,customers,preferred_channel,13,2.00
5,drivers,training_score,7,4.12
6,vehicles,battery_health_pct,4,3.33


In [19]:
processing_summary_df

,dataset,rows_after_cleaning,columns_after_cleaning,missing_values_after_cleaning,duplicate_rows_after_cleaning
0,customers,650,10,0,0
1,orders,1250,11,0,0
2,deliveries,950,14,0,0
3,drivers,170,8,0,0
4,vehicles,120,9,0,0
5,hubs,8,5,0,0
6,complaints,320,12,0,0
7,incidents,280,8,0,0
8,app_events,640,14,0,0


## Interpretation of Python Processing Results

The missing value analysis showed that the highest number of missing values appeared in the app_events dataset, specifically in the order_id column, where 144 records were missing. This indicates that not all mobile application events are directly linked to an order, which reflects the flexible and semi-structured nature of platform-generated event data.

Other missing values were found in operational fields such as booking_channel, loyalty_score, delivery_completed_at, resolved_hours, compensation_amount, customer_rating_post_delivery, preferred_channel, training_score, and battery_health_pct. These fields are important for understanding customer behaviour, delivery completion, service quality, complaint resolution, driver capability, and vehicle condition.

Missing categorical values were replaced with `Unknown`, while missing numerical values were replaced using the median. This approach preserves dataset size while reducing the effect of extreme values. After cleaning, all datasets contained zero missing values, making them suitable for SQL analysis, R analytics, and MongoDB development.

The processing stage also created additional analytical columns such as delay flags, failure flags, complaint severity indicators, and app event issue indicators. These engineered features will support later analysis of service delays, operational failures, customer dissatisfaction, and platform exceptions.